<a href="https://colab.research.google.com/github/micaeladepaolocabarcos/practica-train-test-sobreajuste/blob/main/m8_practica_train_test_sobreajuste.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Práctica: Train/Test y sobreajuste**
**Proyecto:** Producción de pozos de gas y petróleo no convencional (Vaca Muerta, Argentina)

**Fuente:** Secretaría de Energía de la Nación — [datos.gob.ar](https://www.datos.gob.ar/dataset/produccion-de-petroleo-y-gas-por-pozo)

**Objetivo:** evaluar si un modelo aprende patrones o memoriza, comparando su rendimiento en entrenamiento y en prueba.

**Problema:** predecir la producción mensual de petróleo (`prod_pet`, en m³) de pozos petrolíferos no convencionales.

**1. Preparación del entorno**

In [15]:
# Importo las librerías
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

In [16]:
# Defino el nombre del archivo
ARCHIVO = 'produccion-de-pozos-de-gas-y-petrleo-no-convencional.csv'

# Si estoy en Colab conecto Drive; si no, busco el CSV en la misma carpeta del notebook
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_CSV = '/content/drive/MyDrive/Colab Notebooks/' + ARCHIVO
except ImportError:
    RUTA_CSV = ARCHIVO

df = pd.read_csv(RUTA_CSV, low_memory=False)
print('Filas cargadas:', len(df))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Filas cargadas: 421022


In [17]:
# Aplico los filtros de limpieza de las unidades anteriores
# Son filtros fila por fila (no calculan estadísticas), así que no generan fuga de datos
df = df[(df['prod_pet'] >= 0) & (df['prod_gas'] >= 0) & (df['prod_agua'] >= 0)]
df = df[(df['profundidad'] > 0) & (df['profundidad'] <= 8000)]

# Me quedo con pozos petrolíferos en los meses en que estuvieron produciendo
df = df[(df['tipopozo'] == 'Petrolífero') & (df['tipoestado'] == 'Extracción Efectiva')]
print('Filas después de filtrar:', len(df))

Filas después de filtrar: 140200


In [18]:
# Defino las características (X) y el target (y)
caracteristicas = ['profundidad', 'tef', 'prod_agua', 'anio']
X = df[caracteristicas]
y = df['prod_pet']

print('X:', X.shape)
print('y:', y.shape)

X: (140200, 4)
y: (140200,)


**2. División 80/20**

In [19]:
# Separo 80% para entrenar y 20% para probar
# Fijo random_state para que la división sea siempre la misma
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Train:', X_train.shape[0], 'filas')
print('Test:', X_test.shape[0], 'filas')

Train: 112160 filas
Test: 28040 filas


**3. Entrenamiento**

In [20]:
# Creo un árbol de decisión sin límite de profundidad
modelo = DecisionTreeRegressor(random_state=42)

# Entreno el modelo solo con los datos de entrenamiento
modelo.fit(X_train, y_train)

DecisionTreeRegressor(random_state=42)

**4. Evaluación dual**

`.score()` en un regresor devuelve el **R²**: 1 es una predicción perfecta y 0 equivale a predecir siempre el promedio.

In [21]:
# Guardo cada conjunto en un diccionario para evaluarlos con el mismo bucle
conjuntos = {'Train': (X_train, y_train), 'Test': (X_test, y_test)}

# Creo un diccionario vacío para guardar los puntajes
puntajes = {}

# Recorro cada conjunto y calculo su R²
for nombre, (X_conjunto, y_conjunto) in conjuntos.items():
    puntajes[nombre] = modelo.score(X_conjunto, y_conjunto)
    print(f'R² {nombre}: {puntajes[nombre]:.3f}')

# Calculo la diferencia entre ambos
print(f'Diferencia: {puntajes["Train"] - puntajes["Test"]:.3f}')

R² Train: 1.000
R² Test: 0.432
Diferencia: 0.568


**5. Análisis**

**¿Son parecidos?** No. El modelo obtiene un R² de *1,000 en entrenamiento* y de *0,432 en prueba*, una diferencia de casi 0,57.

**Diagnóstico: overfitting.** Un R² perfecto en train indica que el árbol, al no tener límite de profundidad, se fue ramificando hasta memorizar cada fila de entrenamiento, incluido el ruido. Cuando recibe datos que nunca vio, su rendimiento cae a menos de la mitad. No es underfitting, porque el puntaje en train no es bajo.

**Acción propuesta:** simplificar el modelo limitando la profundidad del árbol (parámetro `max_depth`), para que aprenda reglas generales en lugar de casos puntuales. Otras opciones son exigir un mínimo de registros por hoja (`min_samples_leaf`) o sumar variables que expliquen mejor la producción, como la formación o la antigüedad del pozo.

**Limitación:** como el dataset es una serie temporal por pozo, la división aleatoria mezcla meses de un mismo pozo en train y test. Para el proyecto final, una división cronológica (entrenar con años anteriores y probar con los más recientes) daría una evaluación más honesta.